Bfloat16 test

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B", 
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

Loading weights: 100%|██████████| 311/311 [00:13<00:00, 23.26it/s]


In [4]:
import time
import torch
num_iterations = 5
batch_sizes = [4,8,16,32]
for batch_size in batch_sizes:
    prompts = ["The capital of France is" for _ in range(0, batch_size)]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    print(f"{num_iterations} iterations, batch size {len(prompts)}")

    for i in range(num_iterations):
        torch.cuda.synchronize()
        start_time = time.perf_counter()

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        torch.cuda.synchronize()
        end_time = time.perf_counter()


        duration = end_time - start_time
        outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        print(f"Batch size: {batch_size} Iteration {i}: {duration: } seconds")
    

5 iterations, batch size 4


KeyboardInterrupt: 

INT8 Test

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
config = BitsAndBytesConfig(load_in_8bit = True)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B", 
    device_map="auto",
    quantization_config = config
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

In [ ]:
import time
import torch
num_iterations = 5
batch_sizes = [8, 16,32]
for batch_size in batch_sizes:
    prompts = ["The capital of France is" for _ in range(0, batch_size)]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    print(f"{num_iterations} iterations, batch size {len(prompts)}")

    for i in range(num_iterations):
        torch.cuda.synchronize()
        start_time = time.perf_counter()

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        torch.cuda.synchronize()
        end_time = time.perf_counter()

        duration = end_time - start_time
        outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        print(f"Batch size: {batch_size} Iteration {i}: {duration: } seconds")

TO-DO Test GPTQ, AWQ, Native